# XGBoost  Classification
Based on `Feature-engineering.txt`.

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Load the dataset
df = pd.read_csv('../data/attendance_dataset-V2.csv')

## 1. Drop or Transform

In [3]:
# Parse Date
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

# Week_Number: weeks since 22-Jun-2026
start_date = pd.to_datetime('22-06-2026', format='%d-%m-%Y')
df['Week_Number'] = ((df['Date'] - start_date).dt.days // 7) + 1

# Day_Number_of_Semester
df['Day_Number_of_Semester'] = (df['Date'] - start_date).dt.days + 1

# Drop Students_Present
df = df.drop(columns=['Students_Present'])

# Compute Rolling_Avg_3 using Attendance_Percentage over Subject, Date, Lecture_Number
df = df.sort_values(by=['Subject', 'Date', 'Lecture_Number'])
df['Rolling_Avg_3'] = df.groupby('Subject')['Attendance_Percentage'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)
df['Rolling_Avg_3'] = df['Rolling_Avg_3'].fillna(0)

# Convert Attendance_Percentage to a Band (Target for Classifier)
df['Attendance_Band'] = pd.qcut(df['Attendance_Percentage'], q=3, labels=['Low', 'Medium', 'High'])

# Drop Attendance_Percentage and Date
df = df.drop(columns=['Attendance_Percentage', 'Date'])

# Dropping constant columns that are not useful as features
df = df.drop(columns=['End_Time', 'Semester', 'Branch', 'Section', 'Classroom', 'Total_Enrolled'], errors='ignore')


## 2. Keep and Encode

In [4]:
from sklearn.preprocessing import LabelEncoder

# Separate LabelEncoders for each categorical column
le_subject = LabelEncoder()
le_faculty = LabelEncoder()
le_weather = LabelEncoder()

# Day_of_Week
day_map = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6}
df['Day_of_Week'] = df['Day_of_Week'].map(day_map)

# Start_Time: encode as hour (8, 9, 10, 11, 13 etc)
def extract_hour(time_str):
    time_str = time_str.strip()
    parts = time_str.split(' ')
    time_part = parts[0]
    meridian = parts[1] if len(parts) > 1 else ''
    
    # Safely handle both ':' and '.' as separator
    time_part = time_part.replace('.', ':')
    hour = int(time_part.split(':')[0])
    
    if meridian == 'PM' and hour != 12:
        hour += 12
    if meridian == 'AM' and hour == 12:
        hour = 0
    return hour

df['Start_Time'] = df['Start_Time'].apply(extract_hour)

# Subject, Faculty_ID, Weather
df['Subject'] = le_subject.fit_transform(df['Subject'])
df['Faculty_ID'] = le_faculty.fit_transform(df['Faculty_ID'])
df['Weather'] = le_weather.fit_transform(df['Weather'])

# Practical_Theory (Theory=0, Practical=1)
df['Practical_Theory'] = df['Practical_Theory'].map({'Theory': 0, 'Practical': 1})

# Binary encodes
binary_map = {'No': 0, 'Yes': 1}
df['Internal_Test_Week'] = df['Internal_Test_Week'].map(binary_map)
df['Assignment_Due'] = df['Assignment_Due'].map(binary_map)
df['Special_Event'] = df['Special_Event'].map(binary_map)

# Holiday_Before_After (No=0, Before=1, After=2)
holiday_map = {'No': 0, 'Before': 1, 'After': 2}
df['Holiday_Before_After'] = df['Holiday_Before_After'].map(holiday_map)

# Gap_Since_Previous_Lecture (Same Day=0, 1 Day=1, 2 Days=2, etc)
def parse_gap(gap_str):
    if gap_str == 'Same Day':
        return 0
    else:
        return int(gap_str.split(' ')[0])

df['Gap_Since_Previous_Lecture'] = df['Gap_Since_Previous_Lecture'].apply(parse_gap)


## 3. Engineer New Features

In [5]:
# Is_First_Lecture_of_Day
df['Is_First_Lecture_of_Day'] = (df['Lecture_Number'] == 1).astype(int)

# Is_Afternoon (starts at 1.30 PM or later)
df['Is_Afternoon'] = (df['Start_Time'] >= 12).astype(int)

df = df.reset_index(drop=True)

df.head()


,Day_of_Week,Lecture_Number,Start_Time,Subject,Faculty_ID,Previous_Lecture_Attendance,Gap_Since_Previous_Lecture,Practical_Theory,Internal_Test_Week,Assignment_Due,Holiday_Before_After,Weather,Special_Event,Week_Number,Day_Number_of_Semester,Rolling_Avg_3,Attendance_Band,Is_First_Lecture_of_Day,Is_Afternoon
0,1,5,13,0,4,39,2,1,1,0,0,0,0,1,2,15.690000,Medium,0,1
1,1,5,13,0,4,79,7,1,1,0,0,0,1,2,9,19.610000,High,0,1
2,1,5,13,0,4,16,1,1,0,1,1,2,0,3,16,17.486667,Low,0,1
3,1,5,13,0,4,13,2,1,0,1,0,0,0,4,23,17.323333,Medium,0,1
4,1,5,13,0,4,15,2,1,0,0,0,2,1,5,30,14.546667,Medium,0,1


## 4. Train-Test Split (Time-Based)

In [6]:
# To perform a time-based split, we must sort the data purely chronologically.
from sklearn.preprocessing import LabelEncoder
df = df.sort_values(by=['Day_Number_of_Semester', 'Start_Time'])

X = df.drop(columns=['Attendance_Band'])
y = df['Attendance_Band']

# 80-20 Time-based split
train_size = int(len(X) * 0.8)

X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

label_enc = LabelEncoder()
y_train_enc = label_enc.fit_transform(y_train)
y_test_enc  = label_enc.transform(y_test)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")


Train size: 172, Test size: 44


## 5. Random Forest Classifier Training

In [7]:
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_sample_weight


In [8]:
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.compose import ColumnTransformer
# from sklearn.metrics import classification_report, accuracy_score

# We scale only the continuous features, while letting the binary/categorical features passthrough
cols_to_scale = ['Previous_Lecture_Attendance', 'Week_Number', 'Rolling_Avg_3', 'Gap_Since_Previous_Lecture', 'Day_Number_of_Semester']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), cols_to_scale)
    ],
    remainder='passthrough'
)

# Initialize and train the XGBoost Classifier
xgb_clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(n_estimators=100, random_state=42,eval_metric='mlogloss', subsample=0.8, learning_rate=0.1, max_depth=4))
])
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_enc)

xgb_clf.fit(X_train, y_train_enc,  classifier__sample_weight=sample_weights)

# Predict and Evaluate
y_pred = xgb_clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test_enc, y_pred))
print("\nClassification Report:\n", classification_report(y_test_enc, y_pred,target_names=label_enc.classes_))


Accuracy: 0.7272727272727273

Classification Report:
               precision    recall  f1-score   support

        High       1.00      1.00      1.00        15
         Low       0.56      0.64      0.60        14
      Medium       0.62      0.53      0.57        15

    accuracy                           0.73        44
   macro avg       0.73      0.73      0.72        44
weighted avg       0.73      0.73      0.73        44



## Adding the model to the application 

In [9]:
import joblib

joblib.dump(xgb_clf, '../model/xgb-classifier-model.pkl')

['../model/xgb-classifier-model.pkl']

In [10]:
joblib.dump(label_enc,'../model/label_encoder.pkl')
joblib.dump(le_subject,'../model/subject_label_encoder.pkl')


['../model/subject_label_encoder.pkl']